# ClimateTwin V2 — Student Notebook

**AquaAir General Insurance | Full classroom version**

This notebook is intentionally structured so you spend class time on **actuarial judgement**, not routine simulation coding. Core helpers for scenario transformation, policy financial terms, annual-loss simulation, pricing metrics, and repeated-seed tail checks are supplied.

You are responsible for:

- interpreting the insurance contract and data;
- fitting and diagnosing the frequency and severity models;
- explaining important model effects;
- comparing baseline and stress results;
- evaluating a finite set of feasible mitigation candidates; and
- defending a Climate Risk Committee recommendation.

Read `case_study.md` and `data_dictionary.md` before starting.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

DATA_PATH = Path("dataset.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("dataset.csv not found. Keep it in the same folder as this notebook.")

SEED = 20260915
RESILIENCE_BUDGET = 8_000_000
FIXED_EXPENSE = 750.0
VARIABLE_EXPENSE_RATIO = 0.25
PROFIT_CONTINGENCY = 0.05

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Years:", sorted(df["year"].unique()))
print("Unique locations:", df["location_id"].nunique())


## 0. Contract mechanics — do this before modelling

The V2 contract applies the BI waiting period only to BI, then applies the deductible once to total covered loss, then applies the per-claim limit.

A valid covered claim may have zero insurer payment.


In [ ]:
def apply_policy_terms(remediation_loss, daily_bi_loss, interruption_days,
                       waiting_days, deductible, policy_limit):
    ground_up_bi = daily_bi_loss * interruption_days
    covered_bi_days = max(interruption_days - waiting_days, 0)
    covered_bi = daily_bi_loss * covered_bi_days
    ground_up = remediation_loss + ground_up_bi
    covered = remediation_loss + covered_bi
    after_deductible = max(covered - deductible, 0.0)
    paid = min(after_deductible, policy_limit)
    return {
        "ground_up_bi_loss": ground_up_bi,
        "covered_bi_loss": covered_bi,
        "ground_up_loss": ground_up,
        "covered_loss": covered,
        "loss_after_deductible": after_deductible,
        "paid_loss": paid,
    }

# These three checks exactly match the worked examples in case_study.md.
worked_examples = pd.DataFrame({
    "Below deductible": apply_policy_terms(20_000, 0, 0, 2, 50_000, 500_000),
    "Limited payment": apply_policy_terms(900_000, 0, 0, 2, 50_000, 400_000),
    "BI waiting not met": apply_policy_terms(100_000, 30_000, 2, 3, 50_000, 500_000),
}).T
worked_examples


**Check your understanding**

1. Why is the payment zero in the first example?
2. Why is the second payment 400,000 rather than 850,000?
3. In the third example, why is BI not covered while 50,000 is still paid?


## 1. Supplied feature and simulation helpers

Do not rewrite these helpers unless your instructor asks you to. Your job is to use them correctly and explain the actuarial meaning of the outputs.


In [ ]:
def add_model_features(data):
    x = data.copy()
    x["pm25_10"] = x["pm25"] / 10.0
    x["ozone_10"] = x["ozone"] / 10.0
    x["turbidity_2"] = x["turbidity"] / 2.0
    x["wqi_10"] = x["water_quality_index"] / 10.0
    x["temp_5"] = x["temperature_c"] / 5.0
    x["rain_500"] = x["annual_rainfall_mm"] / 500.0
    x["log_insured_value"] = np.log(x["insured_value"])
    x["air_water_stress_interaction"] = x["air_stress_flag"] * x["water_stress_flag"]
    return x


def apply_scenario(portfolio, scenario):
    p = portfolio.copy()
    if scenario in ("Air stress", "Compound stress"):
        p["pm25"] = p["pm25"] + 15.0
        p["ozone"] = p["ozone"] + 10.0
    if scenario in ("Water stress", "Compound stress"):
        p["turbidity"] = p["turbidity"] * 1.60
        p["water_quality_index"] = np.maximum(p["water_quality_index"] - 15.0, 0.0)

    p["air_stress_flag"] = ((p["pm25"] >= 35.0) | (p["ozone"] >= 70.0)).astype(int)
    p["water_stress_flag"] = ((p["turbidity"] >= 5.0) | (p["water_quality_index"] <= 60.0)).astype(int)
    return add_model_features(p)


def predict_portfolio(portfolio, freq_model, sev_model):
    p = add_model_features(portfolio)
    p["mu_count"] = freq_model.predict(p, offset=np.log(p["exposure_years"]))
    p["mu_covered"] = sev_model.predict(p)
    if not np.isfinite(p[["mu_count", "mu_covered"]].to_numpy()).all():
        raise ValueError("Non-finite model predictions. Diagnose the fitted model.")
    return p


def simulate_paid_portfolio(predicted_portfolio, sev_model, n_sims=10_000, seed=SEED):
    """Simulate covered claim severity, then apply deductible and per-claim limit."""
    rng = np.random.default_rng(seed)
    totals = np.zeros(n_sims, dtype=float)

    phi = max(float(sev_model.scale), 1e-9)
    gamma_shape = 1.0 / phi

    for row in predicted_portfolio.itertuples(index=False):
        counts = rng.poisson(max(float(row.mu_count), 0.0), size=n_sims)
        n_claims = int(counts.sum())
        if n_claims == 0:
            continue

        covered = rng.gamma(
            shape=gamma_shape,
            scale=max(float(row.mu_covered), 1e-9) / gamma_shape,
            size=n_claims,
        )
        paid = np.minimum(np.maximum(covered - float(row.deductible), 0.0), float(row.policy_limit))

        sim_index = np.repeat(np.arange(n_sims), counts)
        np.add.at(totals, sim_index, paid)
    return totals


def tail_metrics(losses, alpha=0.99):
    var = float(np.quantile(losses, alpha))
    return {
        "mean_paid_loss": float(np.mean(losses)),
        "VaR_99": var,
        "TVaR_99": float(np.mean(losses[losses >= var])),
    }


def pricing_metrics(losses, portfolio):
    m = tail_metrics(losses)
    exposure = float(portfolio["exposure_years"].sum())
    current_premium = float(portfolio["current_premium"].sum())
    indicated = (m["mean_paid_loss"] + FIXED_EXPENSE * exposure) / (
        1 - VARIABLE_EXPENSE_RATIO - PROFIT_CONTINGENCY
    )
    return {
        **m,
        "current_premium": current_premium,
        "indicated_premium": indicated,
        "adequacy_ratio": current_premium / indicated,
    }


def repeated_tvar(portfolio, freq_model, sev_model, seeds=(4101, 4102, 4103, 4104),
                  n_sims_per_seed=5_000):
    """Monte Carlo stability check; not a parameter/model uncertainty interval."""
    pred = predict_portfolio(portfolio, freq_model, sev_model)
    rows = []
    for seed in seeds:
        losses = simulate_paid_portfolio(pred, sev_model, n_sims_per_seed, seed)
        rows.append({"seed": seed, **tail_metrics(losses)})
    return pd.DataFrame(rows)


## Stage 1 — Observe: coverage and portfolio understanding

### Tasks

- Confirm the grain, years, exposure, occupancy, and regions.
- Compare environmental stress flags with insured trigger flags.
- Calculate observed claim frequency per exposure.
- Summarize annual ground-up, covered, and paid loss.
- Identify at least one portfolio concentration or non-obvious pattern.

Remember: stress flag ≠ insured trigger ≠ claim ≠ payment.


In [ ]:
# TODO: Create a compact portfolio summary and at least one grouped table.
# Suggested groups: region and/or occupancy.

# YOUR CODE HERE


In [ ]:
# TODO: Make one useful visualization for Stage 1.

# YOUR CODE HERE


**TODO — Stage 1 interpretation**

Write 3–5 sentences explaining what the data show about the hazard → trigger → claim → payment chain.


## Stage 2 — Model: exposure-adjusted frequency

Fit a Poisson GLM for `claim_count` with `log(exposure_years)` as an offset. A reasonable starting specification includes occupancy, region, scaled environmental/climate variables, the air–water stress interaction, and mitigation indicators.

Do not globally suppress warnings. Capture or read any model warning and decide whether it is meaningful.


In [ ]:
dfm = add_model_features(df)

# TODO: Define and fit your frequency model.
#
# Example pattern:
# freq_formula = (
#     "claim_count ~ C(occupancy) + C(region) + pm25_10 + ozone_10 "
#     "+ turbidity_2 + wqi_10 + temp_5 + rain_500 + drought_indicator "
#     "+ air_water_stress_interaction + air_filtration + water_treatment "
#     "+ business_continuity_plan"
# )
#
# with warnings.catch_warnings(record=True) as fit_warnings:
#     warnings.simplefilter("always")
#     freq_model = smf.glm(
#         formula=freq_formula,
#         data=dfm,
#         family=sm.families.Poisson(),
#         offset=np.log(dfm["exposure_years"]),
#     ).fit()
#
# print("Converged:", freq_model.converged)
# print("Pearson dispersion:", freq_model.pearson_chi2 / freq_model.df_resid)
# for w in fit_warnings:
#     print(type(w.message).__name__, ":", w.message)

# YOUR CODE HERE


In [ ]:
# TODO: Convert selected coefficients to rate ratios and interpret at least three.
# Also comment on the dispersion diagnostic.

# YOUR CODE HERE


## Stage 3 — Model: conditional covered severity

The annual student table contains legitimate zero insurer payments, so **do not** fit Gamma directly to `aggregate_paid_loss / claim_count`.

For claim-positive policy-years define:

```text
average_covered_severity = aggregate_covered_loss / claim_count
```

This is strictly positive. The reference instructor solution uses the supplementary claim-level `covered_loss`, but this annual-average response gives you the same conceptual separation while keeping the full student exercise compact.

Payment effects of deductible and limit are handled in the supplied simulation helper.


In [ ]:
# TODO: Create the claim-positive severity dataset and fit a Gamma-log model.
#
# Suggested preparation:
# sev_df = dfm.loc[dfm["claim_count"] > 0].copy()
# sev_df["average_covered_severity"] = (
#     sev_df["aggregate_covered_loss"] / sev_df["claim_count"]
# )
# assert (sev_df["average_covered_severity"] > 0).all()
#
# Include insured value, occupancy, environmental/climate variables,
# mitigation variables, and BI waiting period as appropriate.
#
# Do not suppress fitting warnings.

# YOUR CODE HERE


In [ ]:
# TODO: Provide a diagnostic (for example fitted vs deviance residual)
# and interpret at least three multiplicative severity effects.

# YOUR CODE HERE


**TODO — Frequency vs severity interpretation**

Which factors appear to affect frequency and severity differently? Why does that distinction matter for pricing and mitigation?


## Stage 4 — Price: baseline 2026 portfolio

Use only 2026 as the current annual portfolio snapshot.

Your fitted models estimate claim count and covered severity. The supplied simulation helper then applies the deductible and limit **per simulated claim** to estimate insurer-paid loss.


In [ ]:
portfolio_2026 = df.loc[df["year"] == 2026].copy()

# TODO:
# 1. predict the Baseline portfolio with predict_portfolio(...)
# 2. simulate at least 10,000 annual paid-loss outcomes
# 3. use pricing_metrics(...)
# 4. report expected paid loss, indicated premium, adequacy ratio, VaR99, TVaR99

# YOUR CODE HERE


## Stage 5 — Stress: four scenarios and tail-risk uncertainty

Required scenarios:

- Baseline
- Air stress
- Water stress
- Compound stress

Use common seeds when comparing scenarios. Then run `repeated_tvar()` (or an equivalent repeated-seed check) so you can tell whether a TVaR ranking is robust or a simulation near-tie.

The repeated-seed variability measures **Monte Carlo uncertainty**, not model or parameter uncertainty.


In [ ]:
SCENARIOS = ["Baseline", "Air stress", "Water stress", "Compound stress"]

# TODO: Run all four scenarios and create one comparison table with:
# mean paid loss, indicated premium, current premium, adequacy ratio, VaR99, TVaR99.
# Use the SAME main seed and simulation count for every scenario.

# YOUR CODE HERE


In [ ]:
# TODO: For every scenario, run a repeated-seed TVaR stability check.
# Summarize mean TVaR across seeds and the between-seed variation.
# Explain whether the scenario ranking looks stable.

# YOUR CODE HERE


In [ ]:
# TODO: Create one loss-distribution or tail-risk visualization.

# YOUR CODE HERE


### Compound-scenario limitation

Your discussion must state that the V2 DGP treats air-trigger and water-trigger draws as **conditionally independent given the simulated covariates**. Compound non-additivity is introduced explicitly once both triggers are active; the simulator does not contain a copula, latent common shock, or spatial dependence process.


## Stage 6 — Mitigate: compare feasible candidate strategies

Budget: **8,000,000 currency units**.

You are **not** being asked to prove a global optimum. Construct or use a manageable set of defensible candidate portfolios, verify each is feasible, and compare them under Compound stress.

Suggested candidate families include:

- air-filtration priority;
- water-treatment priority;
- BCP priority;
- balanced benefit/cost priority; and
- highest modelled-risk-reduction priority.

Your candidate construction can use a transparent proxy. The **final comparison must use simulated TVaR**, with repeated seeds to assess ranking stability.


In [ ]:
def apply_actions(portfolio, selected_actions):
    """Apply a table with columns location_id and flag to a portfolio copy."""
    p = portfolio.copy().reset_index(drop=True)
    if selected_actions is None or len(selected_actions) == 0:
        return p
    loc_to_idx = {loc: i for i, loc in enumerate(p["location_id"])}
    for r in selected_actions.itertuples(index=False):
        p.loc[loc_to_idx[r.location_id], r.flag] = 1
    return p


def greedy_under_budget(action_table, sort_col, budget=RESILIENCE_BUDGET):
    """Convenience helper for constructing a candidate; not an optimizer."""
    selected = []
    spent = 0.0
    for r in action_table.sort_values(sort_col, ascending=False).itertuples(index=False):
        if spent + r.cost <= budget:
            selected.append(r)
            spent += r.cost
    return pd.DataFrame(selected)


In [ ]:
# TODO: Build an action table for missing filtration, water treatment, and BCP.
# Include location_id, action, flag, cost, and a transparent ranking score/proxy.
#
# Then construct at least four feasible candidate portfolios under the budget.

# YOUR CODE HERE


In [ ]:
# TODO: Evaluate every candidate under Compound stress.
# Report:
# - cost
# - expected paid-loss reduction
# - benefit-cost ratio
# - VaR reduction
# - TVaR reduction
# - post-mitigation adequacy
# - repeated-seed TVaR stability
#
# Use the same seeds across candidates.

# YOUR CODE HERE


**Required decision wording**

Use wording such as:

> “Among the feasible candidate portfolios evaluated under the stated budget, ______ had the lowest estimated TVaR99.”

Do **not** write “we found the optimal portfolio” unless you actually solved the complete optimization problem over the entire feasible action space.

If two strategies are very close relative to repeated-seed Monte Carlo variability, report the ranking as uncertain rather than forcing a false distinction.


## Final Climate Risk Committee recommendation

Complete this section after your analysis.

### 1. Baseline position
- Expected paid loss:
- Indicated premium:
- Premium adequacy:
- VaR99:
- TVaR99:

### 2. Most material stress
- Scenario:
- Change from baseline:
- Why it matters:

### 3. Mitigation candidate decision
- Candidate selected among those evaluated:
- Cost:
- Expected-loss reduction:
- TVaR reduction:
- Ranking stability / Monte Carlo uncertainty:

### 4. Pricing / underwriting recommendation
Choose and defend: **Maintain / Reprice / Mitigate / Restrict / Monitor**

### 5. Limitations
State at least three. One must address the compound scenario's conditional-independence limitation.
